# Lab2.3 — Acces direct la registre cu MMIO

În acest task eliminăm driverul `AxiGPIO` și accesăm direct registrele perifericului AXI GPIO folosind clasa PYNQ `MMIO`.


## Obiective

- înțelegerea conceptului Memory-Mapped I/O;
- obținerea adresei fizice a unui IP din `ip_dict`;
- maparea spațiului de registre cu `MMIO`;
- configurarea registrelor DATA și TRI ale AXI GPIO;
- realizarea legăturii Python → MMIO → AXI-Lite → IP hardware.


## 1. Principiu

Pentru canalul 1 al unui AXI GPIO utilizăm două registre:

| Offset | Registru | Rol |
|---:|---|---|
| `0x00` | DATA | citire/scriere date |
| `0x04` | TRI | direcția pinilor: `1` = input, `0` = output |

Fluxul de acces este:

```text
Python
  ↓
MMIO
  ↓
adresă fizică
  ↓
AXI-Lite
  ↓
registre AXI GPIO
  ↓
PL
```


## 2. Identificarea adreselor


In [ ]:
from pynq.overlays.base import BaseOverlay
from pynq import MMIO
import time

base = BaseOverlay("base.bit")

In [ ]:
def find_ip_name(words, exclude=()):
    for name, info in base.ip_dict.items():
        text = (name + " " + info.get("type", "")).lower()
        if "gpio" in text and any(w in name.lower() for w in words):
            if not any(x in name.lower() for x in exclude):
                return name
    raise RuntimeError(f"GPIO IP not found for {words}")

led_name = find_ip_name(("led",), exclude=("rgb",))
button_name = find_ip_name(("button", "btn"))
switch_name = find_ip_name(("switch", "sw"))

for label, name in (("LED", led_name), ("BUTTON", button_name), ("SWITCH", switch_name)):
    info = base.ip_dict[name]
    print(f"{label:6s}: {name:25s} base={hex(info['phys_addr'])} range={hex(info['addr_range'])}")

## 3. Controlul LED-urilor prin MMIO

Mapăm spațiul de registre al controlerului LED și configurăm canalul ca ieșire.


In [ ]:
led_info = base.ip_dict[led_name]
led_mmio = MMIO(led_info["phys_addr"], led_info["addr_range"])

DATA = 0x00
TRI  = 0x04

led_mmio.write(TRI, 0x00000000)   # outputs
print("TRI =", hex(led_mmio.read(TRI)))

In [ ]:
for value in (0x1, 0x2, 0x4, 0x8, 0xF, 0x0):
    led_mmio.write(DATA, value)
    print(f"DATA = 0x{value:X}")
    time.sleep(0.5)

## 4. Citirea butoanelor și switch-urilor prin MMIO

Pentru intrări, registrul TRI este setat la `1` pe biții utilizați.


In [ ]:
button_info = base.ip_dict[button_name]
switch_info = base.ip_dict[switch_name]

button_mmio = MMIO(button_info["phys_addr"], button_info["addr_range"])
switch_mmio = MMIO(switch_info["phys_addr"], switch_info["addr_range"])

button_mmio.write(TRI, 0xFFFFFFFF)
switch_mmio.write(TRI, 0xFFFFFFFF)

print(f"Buttons  = {button_mmio.read(DATA) & 0xF:04b}")
print(f"Switches = {switch_mmio.read(DATA) & 0x3:02b}")

## 5. Aplicație completă prin registre

Cât timp SW0 este activ, copiem registrul DATA al butoanelor în registrul DATA al LED-urilor.


In [ ]:
while (switch_mmio.read(DATA) & 0x1) == 1:
    led_mmio.write(DATA, button_mmio.read(DATA) & 0xF)
    time.sleep(0.01)

led_mmio.write(DATA, 0)
print("Program oprit.")

## 6. Atenție

`MMIO` oferă acces direct la adrese hardware. În acest laborator folosim **doar adresele și offset-urile cunoscute** ale IP-urilor identificate în `ip_dict`. Accesul la adrese inexistente sau la registre greșite poate bloca sistemul.


## 7. Exercițiu

Realizați cu `MMIO` un running light pe cele patru LED-uri. Nu utilizați `base.leds` și nici `AxiGPIO`.


In [ ]:
# Scrieți aici soluția


## 8. Întrebări

1. Care este diferența dintre o adresă fizică de bază și un offset de registru?
2. De ce registrul TRI trebuie configurat înainte de utilizarea GPIO-ului?
3. Ce magistrală conectează în acest caz procesorul cu registrele perifericului din PL?
4. Ce avantaj și ce risc introduce accesul direct prin MMIO?
